
<div style="text-align: center; line-height: 0; padding-top: 9px;">
  <img
    src="https://databricks.com/wp-content/uploads/2018/03/db-academy-rgb-1200px.png"
    alt="Databricks Learning"
  >
</div>



# Benchmark Evaluation


In this demo, **we will focus on evaluating large language models using a benchmark dataset specific to the task at hand.**

**Learning Objectives:**

*By the end of this demo, you will be able to;*

* Obtain reference/benchmark data set for task-specific LLM evaluation
* Evaluate an LLM's performance on a specific task using task-specific metrics
* Compare relative performance of two LLMs using a benchmark set

## REQUIRED - SELECT CLASSIC COMPUTE
Before executing cells in this notebook, please select your classic compute cluster in the lab. Be aware that **Serverless** is enabled by default.

Follow these steps to select the classic compute cluster:
1. Navigate to the top-right of this notebook and click the drop-down menu to select your cluster. By default, the notebook will use **Serverless**.

2. If your cluster is available, select it and continue to the next cell. If the cluster is not shown:

   - Click **More** in the drop-down.

   - In the **Attach to an existing compute resource** window, use the first drop-down to select your unique cluster.

**NOTE:** If your cluster has terminated, you might need to restart it in order to select it. To do this:

1. Right-click on **Compute** in the left navigation pane and select *Open in new tab*.

2. Find the triangle icon to the right of your compute cluster name and click it.

3. Wait a few minutes for the cluster to start.

4. Once the cluster is running, complete the steps above to select your cluster.

## Requirements

Please review the following requirements before starting the lesson:

* To run this notebook, you need to use one of the following Databricks runtime(s): **15.4.x-cpu-ml-scala2.12**



## Classroom Setup

Install required libraries.

In [0]:
%pip install databricks-sdk rouge_score evaluate textstat mlflow>=3.0 databricks-feature-engineering --upgrade
dbutils.library.restartPython()

Before starting the demo, run the provided classroom setup script. This script will define configuration variables necessary for the demo. Execute the following cell:

In [0]:
%run ../Includes/Classroom-Setup-03

**Other Conventions:**

Throughout this demo, we'll refer to the object `DA`. This object, provided by Databricks Academy, contains variables such as your username, catalog name, schema name, working directory, and dataset locations. Run the code block below to view these details:

In [0]:
print(f"Username:          {DA.username}")
print(f"Catalog Name:      {DA.catalog_name}")
print(f"Schema Name:       {DA.schema_name}")
print(f"Working Directory: {DA.paths.working_dir}")
print(f"Dataset Location:  {DA.paths.datasets}")

## Demo Overview

In this demonstration, we will be evaluating the performance of an AI system designed to summarize text.

The text documents that we will be summarizing are a collection of fictional product reviews for grocery products.

The AI system works as follows:

1. Accepts a text document as input
2. Constructs an LLM prompt using few-shot learning to summarize the text
3. Submits the prompt to an LLM for summarization
4. Returns summarized text

See below for an example of the system.

## Step 1: Setup Models to Use

Next, we will setup the model that will be used for evaluation.

We will use **Databricks Claude 3.7 Sonnet** and **Llama 3.3 70B Instruct** for evaluation.

In [0]:
from databricks.sdk.service.serving import ChatMessage
from databricks.sdk import WorkspaceClient

w = WorkspaceClient()

# Define the first model for summarization
def query_summary_system(input: str) -> str:
    messages = [
        {
            "role": "system",
            "content": "You are an assistant that summarizes text. Given a text input, you need to provide a one-sentence summary. You specialize in summarizing reviews of grocery products. Please keep the reviews in first-person perspective if they're originally written in first person. Do not change the sentiment. Do not create a run-on sentence – be concise."
        },
        { 
            "role": "user", 
            "content": input 
        }
    ]
    messages = [ChatMessage.from_dict(message) for message in messages]
    chat_response = w.serving_endpoints.query(
        name="databricks-claude-3-7-sonnet",
        messages=messages,
        temperature=0.1,
        max_tokens=128
    )

    return chat_response.as_dict()["choices"][0]["message"]["content"]

# Define the second model for summarization
def challenger_query_summary_system(input: str) -> str:
    messages = [
        {
            "role": "system",
            "content": "You are an assistant that summarizes text. Given a text input, you need to provide a one-sentence summary. You specialize in summarizing reviews of grocery products. Please keep the reviews in first-person perspective if they're originally written in first person. Do not change the sentiment. Do not create a run-on sentence – be concise."
        },
        { 
            "role": "user", 
            "content": input 
        }
    ]
    messages = [ChatMessage.from_dict(message) for message in messages]
    chat_response = w.serving_endpoints.query(
        name="databricks-meta-llama-3-3-70b-instruct",
        messages=messages,
        temperature=0.1,
        max_tokens=128
    )

    return chat_response.as_dict()["choices"][0]["message"]["content"]

Let's test the models!

In [0]:
query_summary_system(
    "This is the best frozen pizza I've ever had! Sure, it's not the healthiest, but it tasted just like it was delivered from our favorite pizzeria down the street. The cheese browned nicely and fresh tomatoes are a nice touch, too! I would buy it again despite it's high price. If I could change one thing, I'd made it a little healthier – could we get a gluten-free crust option? My son would love that."
)

In [0]:
challenger_query_summary_system(
    "This is the best frozen pizza I've ever had! Sure, it's not the healthiest, but it tasted just like it was delivered from our favorite pizzeria down the street. The cheese browned nicely and fresh tomatoes are a nice touch, too! I would buy it again despite it's high price. If I could change one thing, I'd made it a little healthier – could we get a gluten-free crust option? My son would love that."
)

To complete this workflow, we'll focus on the following steps:

1. Obtain a benchmark set for evaluating summarization
2. Compute summarization-specific evaluation metrics using the benchmark set
3. Compare performance with another LLM using the benchmark set and evaluation metrics

## Step 2: Benchmark and Reference Sets

As a reminder, our task-specific evaluation metrics (including ROUGE for summarization) require a benchmark set to compute scores.

There are two types of reference/benchmark sets that we can use:

1. Large, generic benchmark sets commonly used across use cases
2. Domain-specific benchmark sets specific to your use case

For this demo, we'll focus on the former.

### Generic Benchmark Set

First, we'll import a generic benchmark set used for evaluating text summarization.

We'll use the data set used in [Benchmarking Large Language Models for News Summarization](https://arxiv.org/abs/2301.13848) to evaluate how well our LLM solution summarizes general text.

This data set:

* is relatively large in scale at 599 records
* is related to news articles
* contains original text and *author-written* summaries of the original text

**Question:** What is the advantage of using ground-truth summaries that are written by the original author?

In [0]:
import pandas as pd

# Read and display the dataset
eval_data = pd.read_csv(f"{DA.paths.datasets.news}/csv/news-summaries.csv")
display(eval_data)

## Step 3: Compute the ROUGE Evaluation Metric

Next, we will want to compute our ROUGE-N metric to understand how well our system summarizes grocery generic text using the benchmark dataset.

We can compute the ROUGE metric (among others) using MLflow's new LLM evaluation capabilities. MLflow LLM evaluation includes default collections of metrics for pre-selected tasks, e.g, “question-answering” or "text-summarization" (our case). Depending on the LLM use case that you are evaluating, these pre-defined collections can greatly simplify the process of running evaluations.

The `mlflow.evaluate` function accepts the following parameters for this use case:

* An LLM model
* Reference data for evaluation (our benchmark set)
* Column with ground truth data
* The model/task type (e.g. `"text-summarization"`)

**Note:** The `text-summarization` type will automatically compute ROUGE-related metrics. For some metrics, additional library installs will be needed – you can see the requirements in the printed output.

In [0]:
# A custom function to iterate through our eval DF
def query_iteration(inputs):
    answers = []

    for index, row in inputs.iterrows():
        completion = query_summary_system(row["inputs"])
        answers.append(completion)

    return answers

# Test query_iteration function – it needs to return a list of output strings
query_iteration(eval_data.head())

In [0]:
import mlflow

# MLflow's `evaluate` with a custom function
results = mlflow.evaluate(
    query_iteration,                      # iterative function from above
    eval_data.head(50),                   # limiting for speed
    targets="writer_summary",             # column with expected or "good" output
    model_type="text-summarization"       # type of model or task
)

We can view the results for individual records by subsetting the handy `.tables` object.

Notice all of the different versions of the ROUGE metric. These are calculated using the HuggingFace `evaluator` library, and the metrics are detailed [here](https://huggingface.co/spaces/evaluate-metric/rouge).

In summary, the descriptions of each metric are below:

* "rouge1": unigram (1-gram) based scoring
* "rouge2": bigram (2-gram) based scoring
* "rougeL": Longest common subsequence based scoring.
* "rougeLSum": splits text using "\n"

In [0]:
display(results.tables["eval_results_table"].head(10))

And we can view summarized (mean, variance, etc.) model-level (rather than record-level) results with the following:

In [0]:
results.metrics

We are also able to review the results in the MLflow Experiment Tracking UI.

### What does good look like?

The ROUGE metrics range between 0 and 1 – where 0 indicates extremely dissimilar text and 1 indicates extremely similar text. However, our interpretation of what is "good" is usually going to be use-case specific. We don't always want a ROUGE score close to 1 because it's likely not reducing the text size too much.

To explore what "good" looks like, let's review a couple of our examples.

In [0]:
import pandas as pd
display(
    pd.DataFrame(
        results.tables["eval_results_table"]
    ).loc[0:1, ["inputs", "outputs", "rouge1/v1/score"]]
)

**Discussion Questions:**
1. How do you interpret the ROUGE-1 score?
2. Do the scores reflect the summarization that you think is best?

## Step 4: Comparing LLM Performance

In practice, we will frequently be comparing LLMs (or larger AI systems) against one another when determining which is the best for our use case. As a result of this, it's important to become familiar with comparing these solutions.

In the below cell, we demonstrate computing the same metrics using the same reference dataset – but this time, we're summarizing using a system that utilizes a different LLM.

**Note:** This time, we're going to read our reference dataset from Delta.

In [0]:
# A compare custom function to iterate through our eval DF
def challenger_query_iteration(inputs):
    answers = []

    for index, row in inputs.iterrows():
        completion = challenger_query_summary_system(row["inputs"])
        answers.append(completion)

    return answers

# Compute challenger results
challenger_results = mlflow.evaluate(
    challenger_query_iteration,           # iterative function from above
    eval_data.head(50),
    targets="writer_summary",             # column with expected or "good" output
    model_type="text-summarization"       # type of model or task
)

Let's take a look at our model-level results.

In [0]:
challenger_results.metrics

And let's compare in the MLflow UI, looking at the experiment's **Chart** tab.

**Note:** We can filter specifically to ROUGE metrics.

### What about other tasks/metrics?

The `mlflow` library contains [a number of LLM task evaluation tools](https://mlflow.org/docs/latest/python_api/mlflow.html#mlflow.evaluate) that we can use in our workflows.


## Conclusion

You should now be able to:

* Obtain reference/benchmark data set for task-specific LLM evaluation
* Evaluate an LLM's performance on a specific task using task-specific metrics
* Compare relative performance of two LLMs using a benchmark set

&copy; 2026 Databricks, Inc. All rights reserved. Apache, Apache Spark, Spark, the Spark Logo, Apache Iceberg, Iceberg, and the Apache Iceberg logo are trademarks of the <a href="https://www.apache.org/" target="_blank">Apache Software Foundation</a>.<br/><br/><a href="https://databricks.com/privacy-policy" target="_blank">Privacy Policy</a> | <a href="https://databricks.com/terms-of-use" target="_blank">Terms of Use</a> | <a href="https://help.databricks.com/" target="_blank">Support</a>